In [2]:
import os
import glob
import h5py
import numpy as np
import torch

from typing import List, Dict, Optional
import matplotlib.pyplot as plt

from __future__ import annotations

import argparse
from pathlib import Path

In [24]:
def get_streamfunc(vx, vy, Lx=1.0, Ly=1.0):
    nx, ny = vx.shape
    dx = Lx / nx
    dy = Ly / ny
    
    kx = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(ny, d=dy)
    KX, KY = np.meshgrid(kx, ky, indexing="ij")
    
    vx_hat = np.fft.fft2(vx)
    vy_hat = np.fft.fft2(vy)
    
    omega_hat = 1j * KX * vy_hat - 1j * KY * vx_hat
    k2 = KX**2 + KY**2
    
    psi_hat = np.zeros_like(omega_hat)
    mask = k2 != 0
    psi_hat[mask] = -omega_hat[mask] / k2[mask]
    
    return np.fft.ifft2(psi_hat).real

In [25]:
def process_simulation(
    sim_idx: int, output_array: np.ndarray, data_idx: int
) -> None:
    """
    Process data from a single simulation.

    Args:
        sim_idx: Simulation index
        output_array: Array to store processed data
        data_idx: Index in the output array for this simulation
    """
    sim_output_dir = os.path.join(
        "outputs_750/diffusion", f"output-{sim_idx:0{3}}"
    )
    files = sorted(glob.glob(os.path.join(sim_output_dir, "*.h5")))

    if not files:
        print(f"No files found in {sim_output_dir}")
        return

    # Read data from h5 files
    for file in files:
        print(file)
        with h5py.File(file, "r") as f:
            # Get the time axis length
            tmax = f["tasks"]["velocity"].shape[0]

            # Copy data to the output array
            for t in range(26):
                output_array[data_idx, t, :, :, 0] = get_streamfunc(f["tasks"]["velocity"][t*4, 0, ...], f["tasks"]["velocity"][t*4, 1, ...])  # psi
                output_array[data_idx, t, :, :, 1] = f["tasks"]["vector potential"][
                    t*4, ...
                ]  # A

def process_all_simulations(indices: List[int]) -> np.ndarray:
    """
    Process data from all simulations and return a combined array.

    Args:
        indices: List of simulation indices to process

    Returns:
        Array containing processed data from all simulations
    """
    # Create the output array
    data = np.zeros((len(indices), 26, 128, 128, 2))

    # Process each simulation
    for idx, sim_idx in enumerate(indices):
        try:
            process_simulation(sim_idx, data, idx)
        except:
            pass

    return data

In [234]:
data = process_all_simulations([i for i in range(1000,1200)])

In [29]:
np.save("data/data_psiA_Re750_diffusion.npy", data)

In [4]:
from src.utils.fourier_utils import create_wavenumbers, compute_derivative, compute_laplacian

kx, ky, kx_first, ky_first = create_wavenumbers(128,128,1,1,"cpu")

In [5]:
def lowpass_filter(field: torch.Tensor, cutoff_min=0.0, cutoff_max=1.0) -> torch.Tensor:
    nx, ny = field.shape[-2:]
    freq_dtype = torch.float32
    fx = torch.fft.fftfreq(nx, device=field.device, dtype=freq_dtype).reshape(
        1, 1, nx, 1
    )
    fy = torch.fft.rfftfreq(ny, device=field.device, dtype=freq_dtype).reshape(
        1, 1, 1, ny // 2 + 1
    )

    mask = ((fx.square() + fy.square()).sqrt() <= cutoff_max) * ((fx.square() + fy.square()).sqrt() >= cutoff_min)

    field_h = torch.fft.rfftn(field, dim=(-2, -1))
    while mask.ndim > field_h.ndim:
        mask = mask.squeeze(0)
    return torch.fft.irfftn(field_h * mask, s=(nx, ny), dim=(-2, -1))

In [209]:
def laplacian(field):
    return torch.fft.irfftn(compute_laplacian(torch.fft.rfftn(field, dim=[0,1]), kx[0][0], ky[0][0]), dim=[0,1], s=(128,128))
    
def ddx(field):
    return torch.fft.irfftn(compute_derivative(torch.fft.rfftn(field, dim=[0,1]), kx_first[0][0]), dim=[0,1], s=(128,128))
    
def ddy(field):
    return torch.fft.irfftn(compute_derivative(torch.fft.rfftn(field, dim=[0,1]), ky_first[0][0]), dim=[0,1], s=(128,128))

# Calculates the right hand side of the induction equation
def rhs(A, psi, dt):
    ux_pred = -ddy(psi)
    uy_pred = ddx(psi)
    
    A_x = ddx(A)
    A_y = ddy(A)
    lap_A = laplacian(A)
    
    return (0.0013333333333 * lap_A - (ux_pred*A_x + uy_pred*A_y))
    # return (0.001 * lap_A - (ux_pred*A_x + uy_pred*A_y))

In [8]:
def reconstruct(initial_A, psi, freq_max=0.2):
    As = [initial_A]
    A = initial_A
    
    n=8
    dt = 0.04/n
    
    for i in range(n*len(psi)):
        i1 = i//n
        i2 = min(i//n + 1, len(psi)-1)
        psi_interp = psi[i1]*(i%n)/n + psi[i2]*(1 - (i%n)/n)

        k1 = rhs(A, psi_interp, dt) * dt
        k2 = rhs(A + 0.5*dt*k1, psi_interp, dt) * dt
        k3 = rhs(A + 0.5*dt*k2, psi_interp, dt) * dt
        k4 = rhs(A + dt*k3, psi_interp, dt) * dt
        A = A + (k1 + 2*k2 + 2*k3 + k4)/6
        A = lowpass_filter(A, cutoff_max=0.16)
        if i % n == n-1:
            As.append(A)
    return np.array(As[:-1])

In [210]:
# Re 750
std_psi = 0.021676693253362907
std_A = 0.0009381390596884123

# # Re 1000
# std_psi = 0.022428618611113046
# std_A = 0.0009708260721581244

In [ ]:
# Generate training data for the post-reconstruction diffusion model

In [235]:
inputs = []
targets = []

for i in range(20):
    psi = (torch.load(f"results/pred_Re750_{i}.pt")[:,:,0].reshape([-1,128,128]).cpu() * std_psi)
    psi_target = (torch.load(f"results/y_Re750_{i}.pt")[:,:,0].reshape([-1,128,128]).cpu() * std_psi)
    A = (torch.load(f"results/y_Re750_{i}.pt")[:,:,1].reshape([-1,128,128]).cpu() * std_A)
    A_recon = reconstruct(A[0], psi)

    inputs.append(np.concatenate([psi.reshape([1,26,16384]), A_recon.reshape([1,26,16384])], axis=0))
    targets.append(np.concatenate([psi_target.reshape([1,26,16384]), A.reshape([1,26,16384])], axis=0))

inputs = np.array(inputs)
targets = np.array(targets)
np.save("data/diffusion/psiA_newA_Re750/val.npy", {"diff_targets": targets, "diff_inputs": inputs})

In [24]:
np.save("data/diffusion/psiA_newA_Re750/test.npy", {"diff_targets": targets, "diff_inputs": inputs})

In [236]:
inputs = []
targets = []

for i in range(20,200):
    psi = (torch.load(f"results/pred_Re750_{i}.pt")[:,:,0].reshape([-1,128,128]).cpu() * std_psi)
    psi_target = (torch.load(f"results/y_Re750_{i}.pt")[:,:,0].reshape([-1,128,128]).cpu() * std_psi)
    A = (torch.load(f"results/y_Re750_{i}.pt")[:,:,1].reshape([-1,128,128]).cpu() * std_A)
    A_recon = reconstruct(A[0], psi)

    inputs.append(np.concatenate([psi.reshape([1,26,16384]), A_recon.reshape([1,26,16384])], axis=0))
    targets.append(np.concatenate([psi_target.reshape([1,26,16384]), A.reshape([1,26,16384])], axis=0))

inputs = np.array(inputs)
targets = np.array(targets)
np.save("data/diffusion/psiA_newA_Re750/train.npy", {"diff_targets": targets, "diff_inputs": inputs})